# Week 16: Introduction to MLOps -- Model Versioning, Inference & Monitoring
# 第 16 週：MLOps 入門 — 模型版本、推論服務與監測

## 學習目標 Learning Objectives
1. 理解 ML 生命週期 (ML Lifecycle) 與 MLOps 核心原則
2. 實作模型序列化 (Serialization) 與版本命名
3. 建立簡易實驗追蹤器 (Experiment Tracker)
4. 訓練多個模型並建立比較儀表板
5. 實作簡易推論 API 概念
6. 使用 KS 檢定偵測資料漂移 (Data Drift)
7. 模擬模型效能降級並建立監測儀表板
8. 實作 A/B 測試統計顯著性檢定
9. 視覺化 CI/CD for ML 流程

**所需套件 Required packages:** numpy, pandas, matplotlib, scikit-learn, scipy, json, pickle, joblib

---

In [ ]:
# === Imports and Setup ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import json
import pickle
import joblib
import os
import time
import tempfile
from datetime import datetime

from sklearn.datasets import load_wine, load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.pipeline import Pipeline
from scipy import stats

# 設定繪圖風格 Setting plot style
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

np.random.seed(42)

print('All packages imported successfully!')
print('\u6240\u6709\u5957\u4ef6\u532f\u5165\u6210\u529f\uff01')

In [ ]:
# === Train a Baseline Model ===
# 訓練基線模型：使用 Wine 資料集
wine = load_wine()
X, y = wine.data, wine.target
feature_names = wine.feature_names
target_names = wine.target_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train baseline RandomForest
baseline_model = RandomForestClassifier(n_estimators=100, random_state=42)
baseline_model.fit(X_train, y_train)

train_acc = baseline_model.score(X_train, y_train)
test_acc = baseline_model.score(X_test, y_test)

print(f'Wine Dataset: {X.shape[0]} samples, {X.shape[1]} features, {len(target_names)} classes')
print(f'Train/Test split: {X_train.shape[0]}/{X_test.shape[0]}')
print(f'Baseline RandomForest - Train Accuracy: {train_acc:.4f}')
print(f'Baseline RandomForest - Test  Accuracy: {test_acc:.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, baseline_model.predict(X_test), target_names=target_names))

---
## Part 1: ML Lifecycle Visualization
## 第一部分：ML 生命週期視覺化

MLOps 的核心是管理機器學習的完整生命週期。從問題定義到模型部署與監測，形成一個持續改善的循環。

The core of MLOps is managing the complete ML lifecycle. From problem definition to model deployment and monitoring, it forms a continuous improvement cycle.

$$
\text{MLOps} = \text{ML} + \text{DevOps} + \text{Data Engineering}
$$

In [ ]:
# === ML Lifecycle Diagram ===
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.axis('off')

# Define lifecycle stages in a circle
stages = [
    ('Problem\nDefinition', '\u554f\u984c\u5b9a\u7fa9'),
    ('Data\nCollection', '\u8cc7\u6599\u6536\u96c6'),
    ('Data\nPreparation', '\u8cc7\u6599\u6e96\u5099'),
    ('Model\nTraining', '\u6a21\u578b\u8a13\u7df4'),
    ('Model\nEvaluation', '\u6a21\u578b\u8a55\u4f30'),
    ('Model\nDeployment', '\u6a21\u578b\u90e8\u7f72'),
    ('Model\nMonitoring', '\u6a21\u578b\u76e3\u6e2c'),
]

n = len(stages)
center = (5, 5)
radius = 3.2
colors = ['#FF6B6B', '#FFA07A', '#FFD700', '#90EE90', '#87CEEB', '#9370DB', '#FF69B4']

# Draw stages as circles
for i, ((eng, chi), color) in enumerate(zip(stages, colors)):
    angle = 90 - i * (360 / n)
    angle_rad = np.radians(angle)
    x = center[0] + radius * np.cos(angle_rad)
    y = center[1] + radius * np.sin(angle_rad)
    
    circle = plt.Circle((x, y), 0.85, color=color, alpha=0.8, ec='white', lw=2)
    ax.add_patch(circle)
    ax.text(x, y + 0.15, eng, ha='center', va='center', fontsize=8, fontweight='bold', color='white',
            path_effects=[pe.withStroke(linewidth=2, foreground='black')])
    ax.text(x, y - 0.4, chi, ha='center', va='center', fontsize=7, color='white',
            path_effects=[pe.withStroke(linewidth=2, foreground='black')])

# Draw arrows between stages
for i in range(n):
    angle1 = 90 - i * (360 / n)
    angle2 = 90 - ((i + 1) % n) * (360 / n)
    a1_rad = np.radians(angle1)
    a2_rad = np.radians(angle2)
    
    x1 = center[0] + radius * np.cos(a1_rad)
    y1 = center[1] + radius * np.sin(a1_rad)
    x2 = center[0] + radius * np.cos(a2_rad)
    y2 = center[1] + radius * np.sin(a2_rad)
    
    # Shorten arrows so they don't overlap circles
    dx = x2 - x1
    dy = y2 - y1
    dist = np.sqrt(dx**2 + dy**2)
    offset = 0.95 / dist
    
    ax.annotate('', xy=(x1 + dx * (1 - offset), y1 + dy * (1 - offset)),
               xytext=(x1 + dx * offset, y1 + dy * offset),
               arrowprops=dict(arrowstyle='->', color='#333', lw=2))

# Center label
ax.text(center[0], center[1] + 0.3, 'ML Lifecycle', ha='center', va='center',
        fontsize=16, fontweight='bold', color='#333')
ax.text(center[0], center[1] - 0.3, 'ML \u751f\u547d\u9031\u671f', ha='center', va='center',
        fontsize=13, color='#555')
ax.text(center[0], center[1] - 1.0, 'Continuous Improvement\n\u6301\u7e8c\u6539\u5584',
        ha='center', va='center', fontsize=10, color='#777', style='italic')

plt.title('ML Lifecycle / ML \u751f\u547d\u9031\u671f\u5716', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print('\u91cd\u9ede Key Takeaway:')
print('ML \u751f\u547d\u9031\u671f\u662f\u4e00\u500b\u5faa\u74b0\u904e\u7a0b\uff0c\u4e0d\u662f\u7dda\u6027\u7684\u3002\u6a21\u578b\u90e8\u7f72\u5f8c\u4ecd\u9700\u6301\u7e8c\u76e3\u6e2c\u8207\u6539\u5584\u3002')
print('The ML lifecycle is a cyclical process, not linear. Continuous monitoring and improvement are needed after deployment.')

---
## Part 2: Model Serialization & Versioning
## 第二部分：模型序列化與版本管理

模型序列化 (Serialization) 是將訓練好的模型儲存為檔案，以便後續載入使用。Python 常用的方式包括 `pickle` 和 `joblib`。

Model serialization converts a trained model into a file that can be stored and loaded later.

**Version naming convention:**
```
model_<name>_v<major>.<minor>_<YYYYMMDD>.pkl
```

In [ ]:
# === Model Serialization: pickle vs joblib ===
tmpdir = tempfile.mkdtemp()

# Save with pickle
pickle_path = os.path.join(tmpdir, 'model_rf_v1.0_20260307.pkl')
t0 = time.time()
with open(pickle_path, 'wb') as f:
    pickle.dump(baseline_model, f)
pickle_save_time = time.time() - t0
pickle_size = os.path.getsize(pickle_path)

# Save with joblib
joblib_path = os.path.join(tmpdir, 'model_rf_v1.0_20260307.joblib')
t0 = time.time()
joblib.dump(baseline_model, joblib_path)
joblib_save_time = time.time() - t0
joblib_size = os.path.getsize(joblib_path)

# Save with joblib (compressed)
joblib_comp_path = os.path.join(tmpdir, 'model_rf_v1.0_20260307_compressed.joblib')
t0 = time.time()
joblib.dump(baseline_model, joblib_comp_path, compress=3)
joblib_comp_save_time = time.time() - t0
joblib_comp_size = os.path.getsize(joblib_comp_path)

# Load and verify
with open(pickle_path, 'rb') as f:
    loaded_pickle = pickle.load(f)
loaded_joblib = joblib.load(joblib_path)

# Verify predictions match
pred_original = baseline_model.predict(X_test)
pred_pickle = loaded_pickle.predict(X_test)
pred_joblib = loaded_joblib.predict(X_test)

print('=== Model Serialization Comparison ===')
print(f'\n{"Method":<25} {"File Size":>12} {"Save Time":>12} {"Predictions Match":>20}')
print('-' * 72)
print(f'{"pickle":<25} {pickle_size/1024:>9.1f} KB {pickle_save_time:>9.4f} s {np.array_equal(pred_original, pred_pickle):>20}')
print(f'{"joblib":<25} {joblib_size/1024:>9.1f} KB {joblib_save_time:>9.4f} s {np.array_equal(pred_original, pred_joblib):>20}')
print(f'{"joblib (compressed)":<25} {joblib_comp_size/1024:>9.1f} KB {joblib_comp_save_time:>9.4f} s {"True":>20}')

# Version naming demo
print('\n=== Version Naming Convention ===')
versions = [
    ('model_rf_v1.0_20260301.joblib', 'Initial model, 100 estimators'),
    ('model_rf_v1.1_20260305.joblib', 'Tuned max_depth=10'),
    ('model_rf_v2.0_20260307.joblib', 'New features added, retrained'),
]
for name, desc in versions:
    print(f'  {name:<45} -- {desc}')

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
methods = ['pickle', 'joblib', 'joblib\n(compressed)']
sizes = [pickle_size/1024, joblib_size/1024, joblib_comp_size/1024]
times = [pickle_save_time * 1000, joblib_save_time * 1000, joblib_comp_save_time * 1000]
bar_colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

axes[0].bar(methods, sizes, color=bar_colors, edgecolor='white', linewidth=2)
axes[0].set_ylabel('File Size (KB)')
axes[0].set_title('Model File Size Comparison')
for i, v in enumerate(sizes):
    axes[0].text(i, v + 1, f'{v:.1f} KB', ha='center', fontweight='bold')

axes[1].bar(methods, times, color=bar_colors, edgecolor='white', linewidth=2)
axes[1].set_ylabel('Save Time (ms)')
axes[1].set_title('Serialization Time Comparison')
for i, v in enumerate(times):
    axes[1].text(i, v + 0.1, f'{v:.2f} ms', ha='center', fontweight='bold')

plt.suptitle('Model Serialization: pickle vs joblib / \u6a21\u578b\u5e8f\u5217\u5316\u6bd4\u8f03', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 3: Model Metadata & Experiment Tracking
## 第三部分：模型元數據與實驗追蹤

在 MLOps 中，每次實驗都應記錄超參數 (Hyperparameters)、指標 (Metrics)、時間戳記等元數據 (Metadata)。

In MLOps, every experiment run should log hyperparameters, metrics, and timestamps.

$$
\text{Experiment Record} = \{\text{params}, \text{metrics}, \text{timestamp}, \text{model\_path}\}
$$

In [ ]:
# === Simple JSON-based Experiment Tracker ===
class SimpleExperimentTracker:
    """A lightweight experiment tracker using JSON files.
    \u7c21\u6613\u5be6\u9a57\u8ffd\u8e64\u5668\uff0c\u4f7f\u7528 JSON \u6a94\u6848\u5132\u5b58\u3002
    """
    def __init__(self, experiment_name):
        self.experiment_name = experiment_name
        self.runs = []
    
    def log_run(self, run_name, params, metrics, notes=''):
        """Log a single experiment run."""
        run_record = {
            'run_name': run_name,
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'params': params,
            'metrics': metrics,
            'notes': notes
        }
        self.runs.append(run_record)
        print(f'  Logged run: {run_name} | Accuracy: {metrics.get("accuracy", "N/A")}')
        return run_record
    
    def get_best_run(self, metric='accuracy'):
        """Get the run with the best value for a given metric."""
        if not self.runs:
            return None
        return max(self.runs, key=lambda r: r['metrics'].get(metric, 0))
    
    def to_dataframe(self):
        """Convert experiment log to a pandas DataFrame."""
        rows = []
        for run in self.runs:
            row = {'run_name': run['run_name'], 'timestamp': run['timestamp']}
            row.update({f'param_{k}': v for k, v in run['params'].items()})
            row.update(run['metrics'])
            rows.append(row)
        return pd.DataFrame(rows)
    
    def save(self, filepath):
        """Save experiment log to JSON."""
        data = {'experiment_name': self.experiment_name, 'runs': self.runs}
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        print(f'  Experiment log saved to {filepath}')

# Create tracker and log experiments
tracker = SimpleExperimentTracker('wine-classification')

# Experiment 1: RandomForest default
tracker.log_run(
    run_name='rf-default',
    params={'model': 'RandomForest', 'n_estimators': 100, 'max_depth': None},
    metrics={'accuracy': test_acc, 'train_accuracy': train_acc},
    notes='Baseline RandomForest'
)

# Experiment 2: RandomForest tuned
rf_tuned = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
rf_tuned.fit(X_train, y_train)
tracker.log_run(
    run_name='rf-tuned',
    params={'model': 'RandomForest', 'n_estimators': 200, 'max_depth': 10},
    metrics={'accuracy': rf_tuned.score(X_test, y_test), 'train_accuracy': rf_tuned.score(X_train, y_train)},
    notes='Tuned RandomForest'
)

# Experiment 3: LogisticRegression
lr_model = LogisticRegression(max_iter=2000, random_state=42)
lr_model.fit(X_train, y_train)
tracker.log_run(
    run_name='logreg',
    params={'model': 'LogisticRegression', 'max_iter': 2000, 'C': 1.0},
    metrics={'accuracy': lr_model.score(X_test, y_test), 'train_accuracy': lr_model.score(X_train, y_train)},
    notes='Logistic Regression baseline'
)

# Display as DataFrame
print('\n=== Experiment Log ===')
df_log = tracker.to_dataframe()
print(df_log.to_string(index=False))

# Show best run
best = tracker.get_best_run('accuracy')
print(f'\nBest run: {best["run_name"]} with accuracy = {best["metrics"]["accuracy"]:.4f}')

# Save to temp JSON
json_path = os.path.join(tmpdir, 'experiment_log.json')
tracker.save(json_path)

# Show JSON content
with open(json_path, 'r') as f:
    print('\nJSON content (first run):')
    data = json.load(f)
    print(json.dumps(data['runs'][0], indent=2))

---
## Part 4: Model Comparison Dashboard
## 第四部分：模型比較儀表板

訓練多個模型並比較它們的效能，是模型選擇 (Model Selection) 的核心步驟。

Training multiple models and comparing their performance is a core step in model selection.

In [ ]:
# === Model Comparison Dashboard ===
models = {
    'LogisticReg': LogisticRegression(max_iter=2000, random_state=42),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42),
    'MLP': MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=1000, random_state=42),
}

results = []
for name, model in models.items():
    # Use pipeline with scaler
    pipe = Pipeline([('scaler', StandardScaler()), ('model', model)])
    pipe.fit(X_train, y_train)
    
    train_score = pipe.score(X_train, y_train)
    test_score = pipe.score(X_test, y_test)
    y_pred = pipe.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    # Cross-validation
    cv_scores = cross_val_score(pipe, X, y, cv=5, scoring='accuracy')
    
    results.append({
        'Model': name,
        'Train Acc': train_score,
        'Test Acc': test_score,
        'F1 Score': f1,
        'CV Mean': cv_scores.mean(),
        'CV Std': cv_scores.std(),
    })

df_results = pd.DataFrame(results).sort_values('Test Acc', ascending=False)
print('=== Model Comparison Results ===')
print(df_results.to_string(index=False, float_format='%.4f'))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart - Test Accuracy
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(df_results)))
bars = axes[0].barh(df_results['Model'], df_results['Test Acc'], color=colors, edgecolor='white', height=0.6)
axes[0].set_xlabel('Test Accuracy')
axes[0].set_title('Model Comparison: Test Accuracy\n\u6a21\u578b\u6bd4\u8f03\uff1a\u6e2c\u8a66\u6e96\u78ba\u7387')
axes[0].set_xlim(0.85, 1.01)
for bar, val in zip(bars, df_results['Test Acc']):
    axes[0].text(val + 0.002, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontweight='bold', fontsize=9)

# CV scores with error bars
axes[1].barh(df_results['Model'], df_results['CV Mean'], xerr=df_results['CV Std'],
             color=colors, edgecolor='white', height=0.6, capsize=3)
axes[1].set_xlabel('CV Accuracy (mean \u00b1 std)')
axes[1].set_title('Cross-Validation Scores\n\u4ea4\u53c9\u9a57\u8b49\u5206\u6578')
axes[1].set_xlim(0.85, 1.01)

plt.suptitle('Model Comparison Dashboard / \u6a21\u578b\u6bd4\u8f03\u5100\u8868\u677f', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 5: Inference API Concept
## 第五部分：推論 API 概念

推論服務 (Inference Service) 讓訓練好的模型能夠透過 API 接受新資料並回傳預測結果。

An inference service allows a trained model to accept new data via API and return predictions.

$$
\text{API}: \mathbf{x}_{\text{new}} \xrightarrow{\text{model}} \hat{y}, P(\hat{y})
$$

In [ ]:
# === Simple Inference Function ===
class ModelServer:
    """A simple model serving class that simulates an inference API.
    \u7c21\u6613\u6a21\u578b\u670d\u52d9\u985e\u5225\uff0c\u6a21\u64ec\u63a8\u8ad6 API\u3002
    """
    def __init__(self, model_path, model_name='default', version='1.0'):
        self.model = joblib.load(model_path)
        self.model_name = model_name
        self.version = version
        self.request_count = 0
        self.class_names = load_wine().target_names.tolist()
        print(f'Model Server initialized: {model_name} v{version}')
    
    def predict(self, features):
        """Make a prediction with confidence scores."""
        self.request_count += 1
        features = np.array(features).reshape(1, -1)
        
        prediction = self.model.predict(features)[0]
        probabilities = self.model.predict_proba(features)[0]
        
        response = {
            'prediction': self.class_names[prediction],
            'predicted_class': int(prediction),
            'confidence': float(probabilities.max()),
            'probabilities': {name: round(float(p), 4) for name, p in zip(self.class_names, probabilities)},
            'model_name': self.model_name,
            'model_version': self.version,
            'request_id': self.request_count
        }
        return response
    
    def health_check(self):
        return {'status': 'healthy', 'model': self.model_name, 'version': self.version,
                'total_requests': self.request_count}

# Initialize server
server = ModelServer(joblib_path, model_name='wine-classifier', version='1.0')

# Simulate API calls
print('\n=== Simulating API Calls ===')
test_samples = X_test[:5]
for i, sample in enumerate(test_samples):
    response = server.predict(sample.tolist())
    true_label = target_names[y_test[i]]
    print(f'\nRequest #{response["request_id"]}:')
    print(f'  Predicted: {response["prediction"]} (confidence: {response["confidence"]:.4f})')
    print(f'  True Label: {true_label}')
    print(f'  Probabilities: {response["probabilities"]}')

# Health check
print(f'\nHealth Check: {server.health_check()}')

---
## Part 6: Data Drift Detection
## 第六部分：資料漂移偵測

資料漂移 (Data Drift) 是指生產環境中輸入資料的分布與訓練資料不同。

Data drift occurs when the distribution of input features in production differs from training data.

**Kolmogorov-Smirnov (KS) Test:**
$$
D = \sup_x |F_{\text{train}}(x) - F_{\text{prod}}(x)|
$$

If $p < 0.05$, we conclude the distributions are significantly different (drift detected).

In [ ]:
# === Simulate Data Drift & KS-Test Detection ===
np.random.seed(42)

# Original training data distribution (feature 0: Alcohol)
train_feature = X_train[:, 0]  # Alcohol content

# Simulate production data with drift (shifted mean)
drift_amount = 0.8  # Shift by 0.8 standard deviations
prod_feature_drifted = train_feature + np.random.normal(drift_amount, 0.3, len(train_feature))

# Also create a no-drift scenario
prod_feature_no_drift = train_feature + np.random.normal(0, 0.1, len(train_feature))

# KS-test
ks_stat_drift, p_val_drift = stats.ks_2samp(train_feature, prod_feature_drifted)
ks_stat_no_drift, p_val_no_drift = stats.ks_2samp(train_feature, prod_feature_no_drift)

print('=== KS-Test Results for Data Drift Detection ===')
print(f'\nFeature: Alcohol (feature index 0)')
print(f'\n  Scenario 1 (With Drift):')
print(f'    KS Statistic: {ks_stat_drift:.4f}')
print(f'    p-value:      {p_val_drift:.6f}')
print(f'    Drift Detected: {"YES" if p_val_drift < 0.05 else "NO"}')
print(f'\n  Scenario 2 (No Drift):')
print(f'    KS Statistic: {ks_stat_no_drift:.4f}')
print(f'    p-value:      {p_val_no_drift:.6f}')
print(f'    Drift Detected: {"YES" if p_val_no_drift < 0.05 else "NO"}')

# Visualize distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# With drift
axes[0].hist(train_feature, bins=25, alpha=0.6, color='#2196F3', label='Training Data', density=True)
axes[0].hist(prod_feature_drifted, bins=25, alpha=0.6, color='#F44336', label='Production (Drifted)', density=True)
axes[0].set_title(f'With Drift: KS={ks_stat_drift:.3f}, p={p_val_drift:.4f}\n\u6709\u6f02\u79fb', fontsize=12)
axes[0].set_xlabel('Alcohol')
axes[0].set_ylabel('Density')
axes[0].legend()
axes[0].axvline(x=train_feature.mean(), color='#2196F3', linestyle='--', alpha=0.8)
axes[0].axvline(x=prod_feature_drifted.mean(), color='#F44336', linestyle='--', alpha=0.8)

# Without drift
axes[1].hist(train_feature, bins=25, alpha=0.6, color='#2196F3', label='Training Data', density=True)
axes[1].hist(prod_feature_no_drift, bins=25, alpha=0.6, color='#4CAF50', label='Production (No Drift)', density=True)
axes[1].set_title(f'No Drift: KS={ks_stat_no_drift:.3f}, p={p_val_no_drift:.4f}\n\u7121\u6f02\u79fb', fontsize=12)
axes[1].set_xlabel('Alcohol')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.suptitle('Data Drift Detection with KS-Test / \u4f7f\u7528 KS \u6aa2\u5b9a\u5075\u6e2c\u8cc7\u6599\u6f02\u79fb', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# === Multi-Feature Drift Detection ===
np.random.seed(42)

# Simulate drift on multiple features
n_features = X_train.shape[1]
drift_results = []

for i in range(n_features):
    train_feat = X_train[:, i]
    # Add random drift to some features
    if i in [0, 3, 6, 9]:  # These features have drift
        shift = np.random.uniform(0.5, 1.5)
        prod_feat = train_feat + np.random.normal(shift, 0.3, len(train_feat))
    else:
        prod_feat = train_feat + np.random.normal(0, 0.05, len(train_feat))
    
    ks_stat, p_val = stats.ks_2samp(train_feat, prod_feat)
    drift_results.append({
        'Feature': feature_names[i],
        'KS Statistic': ks_stat,
        'p-value': p_val,
        'Drift': 'YES' if p_val < 0.05 else 'NO'
    })

df_drift = pd.DataFrame(drift_results)
print('=== Multi-Feature Drift Detection Results ===')
print(df_drift.to_string(index=False))

# Visualize as heatmap-style bar chart
fig, ax = plt.subplots(figsize=(12, 5))
colors_bar = ['#F44336' if d == 'YES' else '#4CAF50' for d in df_drift['Drift']]
bars = ax.bar(range(n_features), df_drift['KS Statistic'], color=colors_bar, edgecolor='white')
ax.axhline(y=0.2, color='gray', linestyle='--', alpha=0.5, label='Alert threshold')
ax.set_xticks(range(n_features))
ax.set_xticklabels([f[:8] for f in feature_names], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('KS Statistic')
ax.set_title('Feature Drift Detection Dashboard / \u7279\u5fb5\u6f02\u79fb\u5075\u6e2c\u5100\u8868\u677f', fontsize=14, fontweight='bold')

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#F44336', label='Drift Detected'),
                   Patch(facecolor='#4CAF50', label='No Drift')]
ax.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.show()

n_drifted = sum(1 for d in df_drift['Drift'] if d == 'YES')
print(f'\nSummary: {n_drifted}/{n_features} features show significant drift')

---
## Part 7: Model Performance Monitoring
## 第七部分：模型效能監測

模型上線後，效能會隨資料分布變化而下降，這稱為**模型衰退 (Model Decay)**。

After deployment, model performance degrades as data distribution changes -- this is called **Model Decay**.

$$
\text{Accuracy}(t) = \text{Accuracy}_0 \cdot e^{-\lambda t} + \epsilon(t)
$$

In [ ]:
# === Simulate Model Performance Degradation ===
np.random.seed(42)

# Simulate 52 weeks of model monitoring
n_weeks = 52
weeks = np.arange(1, n_weeks + 1)

# Baseline accuracy with gradual degradation
initial_accuracy = 0.97
decay_rate = 0.003
noise = np.random.normal(0, 0.01, n_weeks)

# Accuracy degrades over time with some noise
accuracy_over_time = initial_accuracy - decay_rate * weeks + noise
accuracy_over_time = np.clip(accuracy_over_time, 0.7, 1.0)

# Simulate a sudden drift event at week 30
accuracy_over_time[29:] -= 0.05
accuracy_over_time = np.clip(accuracy_over_time, 0.7, 1.0)

# Simulate data drift score (KS statistic)
drift_score = 0.05 + 0.002 * weeks + np.random.normal(0, 0.01, n_weeks)
drift_score[29:] += 0.15  # Sudden drift event
drift_score = np.clip(drift_score, 0, 1)

# Simulate prediction volume
pred_volume = 1000 + 50 * weeks + np.random.randint(-100, 100, n_weeks)

# Create monitoring dashboard
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (1) Accuracy over time
ax1 = axes[0, 0]
ax1.plot(weeks, accuracy_over_time, 'b-', linewidth=2, label='Model Accuracy')
ax1.axhline(y=0.90, color='orange', linestyle='--', label='Warning Threshold (0.90)')
ax1.axhline(y=0.85, color='red', linestyle='--', label='Critical Threshold (0.85)')
ax1.axvline(x=30, color='gray', linestyle=':', alpha=0.5)
ax1.annotate('Drift Event!', xy=(30, accuracy_over_time[29]), xytext=(35, 0.95),
            arrowprops=dict(arrowstyle='->', color='red'), fontsize=10, color='red')
ax1.fill_between(weeks, 0.85, accuracy_over_time, where=accuracy_over_time < 0.90,
                alpha=0.2, color='orange')
ax1.fill_between(weeks, 0, accuracy_over_time, where=accuracy_over_time < 0.85,
                alpha=0.2, color='red')
ax1.set_ylabel('Accuracy')
ax1.set_title('Model Accuracy Over Time / \u6a21\u578b\u6e96\u78ba\u7387\u8b8a\u5316')
ax1.legend(fontsize=8)
ax1.set_ylim(0.75, 1.0)

# (2) Drift Score
ax2 = axes[0, 1]
ax2.plot(weeks, drift_score, 'r-', linewidth=2, label='Drift Score (KS)')
ax2.axhline(y=0.15, color='orange', linestyle='--', label='Warning (0.15)')
ax2.axhline(y=0.25, color='red', linestyle='--', label='Critical (0.25)')
ax2.fill_between(weeks, drift_score, alpha=0.3, color='red')
ax2.set_ylabel('KS Statistic')
ax2.set_title('Data Drift Score / \u8cc7\u6599\u6f02\u79fb\u5206\u6578')
ax2.legend(fontsize=8)

# (3) Prediction Volume
ax3 = axes[1, 0]
ax3.bar(weeks, pred_volume, color='#4ECDC4', alpha=0.7, width=0.8)
ax3.set_xlabel('Week')
ax3.set_ylabel('Predictions/Week')
ax3.set_title('Prediction Volume / \u9810\u6e2c\u91cf')

# (4) Accuracy vs Drift correlation
ax4 = axes[1, 1]
scatter = ax4.scatter(drift_score, accuracy_over_time, c=weeks, cmap='viridis',
                      s=50, alpha=0.7, edgecolors='white')
plt.colorbar(scatter, ax=ax4, label='Week')
ax4.set_xlabel('Drift Score (KS)')
ax4.set_ylabel('Accuracy')
ax4.set_title('Accuracy vs Drift / \u6e96\u78ba\u7387 vs \u6f02\u79fb')

# Add correlation
corr = np.corrcoef(drift_score, accuracy_over_time)[0, 1]
ax4.text(0.05, 0.95, f'Correlation: {corr:.3f}', transform=ax4.transAxes,
        fontsize=11, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

for ax in axes.flat:
    ax.grid(True, alpha=0.3)

plt.suptitle('Model Monitoring Dashboard / \u6a21\u578b\u76e3\u6e2c\u5100\u8868\u677f', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print('\u91cd\u9ede Key Insight:')
print(f'Correlation between drift and accuracy: {corr:.3f}')
print('Higher drift scores strongly correlate with lower accuracy, confirming the need for continuous monitoring.')

---
## Part 8: A/B Testing for Model Comparison
## 第八部分：A/B 測試模型比較

A/B 測試將流量分配給兩個模型版本，比較它們的線上效能。

A/B testing splits traffic between two model versions to compare their online performance.

We use a **two-proportion z-test** to determine statistical significance:

$$
z = \frac{\hat{p}_B - \hat{p}_A}{\sqrt{\hat{p}(1-\hat{p})\left(\frac{1}{n_A}+\frac{1}{n_B}\right)}}
$$

where $\hat{p} = \frac{x_A + x_B}{n_A + n_B}$ is the pooled proportion.

In [ ]:
# === A/B Testing Simulation ===
np.random.seed(42)

# Model A (current production model): 92% accuracy
# Model B (challenger model): 95% accuracy
n_requests_A = 5000
n_requests_B = 5000
true_accuracy_A = 0.92
true_accuracy_B = 0.95

# Simulate outcomes
results_A = np.random.binomial(1, true_accuracy_A, n_requests_A)
results_B = np.random.binomial(1, true_accuracy_B, n_requests_B)

# Observed accuracies
obs_acc_A = results_A.mean()
obs_acc_B = results_B.mean()

# Two-proportion z-test
successes_A = results_A.sum()
successes_B = results_B.sum()
p_pooled = (successes_A + successes_B) / (n_requests_A + n_requests_B)
se = np.sqrt(p_pooled * (1 - p_pooled) * (1/n_requests_A + 1/n_requests_B))
z_stat = (obs_acc_B - obs_acc_A) / se
p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))

print('=== A/B Test Results ===')
print(f'\nModel A (Current):    n={n_requests_A}, accuracy={obs_acc_A:.4f}')
print(f'Model B (Challenger): n={n_requests_B}, accuracy={obs_acc_B:.4f}')
print(f'\nDifference: {obs_acc_B - obs_acc_A:.4f}')
print(f'Z-statistic: {z_stat:.4f}')
print(f'P-value: {p_value:.6f}')
print(f'\nSignificant at alpha=0.05? {"YES" if p_value < 0.05 else "NO"}')
if p_value < 0.05:
    print('Recommendation: Deploy Model B to production!')

# Cumulative accuracy over time (simulate streaming results)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (1) Cumulative accuracy convergence
cum_acc_A = np.cumsum(results_A) / np.arange(1, n_requests_A + 1)
cum_acc_B = np.cumsum(results_B) / np.arange(1, n_requests_B + 1)

axes[0].plot(cum_acc_A, color='#FF6B6B', alpha=0.8, label=f'Model A (final: {obs_acc_A:.4f})')
axes[0].plot(cum_acc_B, color='#4ECDC4', alpha=0.8, label=f'Model B (final: {obs_acc_B:.4f})')
axes[0].axhline(y=true_accuracy_A, color='#FF6B6B', linestyle='--', alpha=0.4)
axes[0].axhline(y=true_accuracy_B, color='#4ECDC4', linestyle='--', alpha=0.4)
axes[0].set_xlabel('Number of Requests')
axes[0].set_ylabel('Cumulative Accuracy')
axes[0].set_title('A/B Test: Cumulative Accuracy\n\u7d2f\u7a4d\u6e96\u78ba\u7387\u8b8a\u5316')
axes[0].legend()
axes[0].set_ylim(0.88, 0.98)
axes[0].grid(True, alpha=0.3)

# (2) Distribution of daily accuracy (simulated batches)
batch_size = 100
n_batches = n_requests_A // batch_size
batch_acc_A = [results_A[i*batch_size:(i+1)*batch_size].mean() for i in range(n_batches)]
batch_acc_B = [results_B[i*batch_size:(i+1)*batch_size].mean() for i in range(n_batches)]

axes[1].hist(batch_acc_A, bins=15, alpha=0.6, color='#FF6B6B', label='Model A', density=True)
axes[1].hist(batch_acc_B, bins=15, alpha=0.6, color='#4ECDC4', label='Model B', density=True)
axes[1].axvline(x=obs_acc_A, color='#FF6B6B', linestyle='--', linewidth=2)
axes[1].axvline(x=obs_acc_B, color='#4ECDC4', linestyle='--', linewidth=2)
axes[1].set_xlabel('Batch Accuracy')
axes[1].set_ylabel('Density')
axes[1].set_title(f'Batch Accuracy Distribution (p={p_value:.4f})\n\u6279\u6b21\u6e96\u78ba\u7387\u5206\u5e03')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('A/B Testing: Model A vs Model B / A/B \u6e2c\u8a66\u6a21\u578b\u6bd4\u8f03', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 9: CI/CD for ML Pipeline Visualization
## 第九部分：CI/CD for ML 流程圖

ML 的 CI/CD 流程不僅包含程式碼的持續整合與交付，還包括持續訓練 (CT) 與持續監測 (CM)。

CI/CD for ML goes beyond code integration and delivery -- it includes Continuous Training (CT) and Continuous Monitoring (CM).

| 流程 | 英文 | 說明 |
|------|------|------|
| CI | Continuous Integration | 程式碼/資料變更觸發自動測試 |
| CT | Continuous Training | 資料更新觸發自動重新訓練 |
| CD | Continuous Delivery | 模型驗證通過後自動部署 |
| CM | Continuous Monitoring | 持續監測模型效能與資料品質 |

In [ ]:
# === CI/CD for ML Pipeline Diagram ===
fig, ax = plt.subplots(1, 1, figsize=(16, 8))
ax.set_xlim(0, 16)
ax.set_ylim(0, 8)
ax.axis('off')

# Pipeline stages
pipeline_stages = [
    (1.5, 6, 'Code / Data\nChange', '#FF6B6B', 'CI Trigger'),
    (4.5, 6, 'Auto\nTesting', '#FFA07A', 'Unit + Data\nValidation'),
    (7.5, 6, 'Auto\nTraining', '#FFD700', 'CT'),
    (10.5, 6, 'Model\nValidation', '#90EE90', 'Performance\nCheck'),
    (13.5, 6, 'Auto\nDeploy', '#87CEEB', 'CD'),
]

# Draw pipeline boxes
for x, y, label, color, sublabel in pipeline_stages:
    box = FancyBboxPatch((x - 1.1, y - 0.8), 2.2, 1.6,
                         boxstyle='round,pad=0.15', facecolor=color, edgecolor='white',
                         linewidth=2, alpha=0.85)
    ax.add_patch(box)
    ax.text(x, y + 0.15, label, ha='center', va='center', fontsize=10, fontweight='bold')
    ax.text(x, y - 0.55, sublabel, ha='center', va='center', fontsize=7, color='#444', style='italic')

# Draw arrows between stages
for i in range(len(pipeline_stages) - 1):
    x1 = pipeline_stages[i][0] + 1.2
    x2 = pipeline_stages[i+1][0] - 1.2
    ax.annotate('', xy=(x2, 6), xytext=(x1, 6),
               arrowprops=dict(arrowstyle='->', color='#333', lw=2.5))

# Monitoring feedback loop (bottom)
monitor_box = FancyBboxPatch((5.9, 2.0), 4.2, 1.4,
                              boxstyle='round,pad=0.15', facecolor='#9370DB',
                              edgecolor='white', linewidth=2, alpha=0.85)
ax.add_patch(monitor_box)
ax.text(8, 2.9, 'Continuous Monitoring (CM)', ha='center', va='center',
        fontsize=11, fontweight='bold', color='white')
ax.text(8, 2.4, '\u6301\u7e8c\u76e3\u6e2c\uff1a\u6548\u80fd + \u6f02\u79fb + \u8cc7\u6599\u54c1\u8cea', ha='center', va='center',
        fontsize=8, color='white')

# Feedback arrows from monitoring back to training
ax.annotate('', xy=(13.5, 5.1), xytext=(13.5, 3.5),
           arrowprops=dict(arrowstyle='->', color='#9370DB', lw=2, connectionstyle='arc3,rad=0'))
ax.annotate('', xy=(1.5, 5.1), xytext=(5.8, 2.7),
           arrowprops=dict(arrowstyle='->', color='#9370DB', lw=2, connectionstyle='arc3,rad=-0.3'))
ax.text(14.5, 4.3, 'Deploy\u2192Monitor', fontsize=8, color='#9370DB', rotation=90, ha='center')
ax.text(3, 3.5, 'Retrain\nTrigger', fontsize=8, color='#9370DB', ha='center')

# Title and labels
ax.text(8, 7.5, 'CI/CD Pipeline for ML / ML \u7684 CI/CD \u6d41\u7a0b', ha='center', va='center',
        fontsize=16, fontweight='bold')

# Legend box at bottom
legend_items = [
    ('CI: Continuous Integration', '#FF6B6B'),
    ('CT: Continuous Training', '#FFD700'),
    ('CD: Continuous Delivery', '#87CEEB'),
    ('CM: Continuous Monitoring', '#9370DB'),
]
for i, (text, color) in enumerate(legend_items):
    ax.add_patch(FancyBboxPatch((1 + i * 3.7, 0.3), 0.3, 0.3,
                                boxstyle='round,pad=0.05', facecolor=color, edgecolor='white'))
    ax.text(1.5 + i * 3.7, 0.45, text, fontsize=8, va='center')

plt.tight_layout()
plt.show()

print('\u91cd\u9ede Key Takeaway:')
print('ML CI/CD \u4e0d\u50c5\u5305\u542b\u7a0b\u5f0f\u78bc\u7684\u6301\u7e8c\u6574\u5408\uff0c\u9084\u5305\u542b\u8cc7\u6599\u9a57\u8b49\u3001\u81ea\u52d5\u8a13\u7df4\u8207\u6301\u7e8c\u76e3\u6e2c\u3002')
print('ML CI/CD goes beyond code CI -- it includes data validation, auto-training, and continuous monitoring.')

---
## Exercises / 練習題

### Exercise 1: Build an Experiment Tracker Class
### 練習 1：建立實驗追蹤器類別

Extend the `SimpleExperimentTracker` with:
- `compare_runs(run_name_1, run_name_2)` method
- `plot_metrics()` method that visualizes all runs
- Loading from a saved JSON file

In [ ]:
# === Exercise 1: Extended Experiment Tracker ===
class ExtendedTracker(SimpleExperimentTracker):
    def compare_runs(self, name1, name2):
        """Compare two runs side by side."""
        # TODO: Find runs by name and print comparison table
        # Hint: Use self.runs to find runs matching name1 and name2
        # Print their params and metrics side by side
        pass
    
    def plot_metrics(self, metric='accuracy'):
        """Plot metrics across all runs as a bar chart."""
        # TODO: Extract the specified metric from each run
        # Create a bar chart comparing all runs
        # Hint: Use self.runs and matplotlib
        pass
    
    @classmethod
    def load(cls, filepath):
        """Load experiment tracker from JSON file."""
        # TODO: Read JSON file and reconstruct tracker
        # Hint: json.load(), then set self.experiment_name and self.runs
        pass

# Test your implementation
# ext_tracker = ExtendedTracker('test-experiment')
# ... log some runs ...
# ext_tracker.compare_runs('rf-default', 'rf-tuned')
# ext_tracker.plot_metrics('accuracy')
print('Exercise 1: Implement the ExtendedTracker class methods above.')

### Exercise 2: Simulate and Detect Concept Drift
### 練習 2：模擬並偵測概念漂移

Concept drift means the relationship $P(Y|X)$ changes over time.

Simulate:
1. Train a model on data where feature X1 > 0 predicts class 1
2. Gradually change the decision boundary over time
3. Monitor and plot accuracy degradation

In [ ]:
# === Exercise 2: Concept Drift Simulation ===
np.random.seed(42)

# TODO: Step 1 - Generate initial training data
# Create 2D data where class depends on X1 (e.g., X1 > 0 => class 1)
# n_samples = 500
# X_sim = np.random.randn(n_samples, 2)
# y_sim = (X_sim[:, 0] > 0).astype(int)  # Initial boundary: X1 = 0

# TODO: Step 2 - Train a model on the initial data
# model_sim = LogisticRegression()
# model_sim.fit(X_sim, y_sim)

# TODO: Step 3 - Simulate concept drift over 20 time steps
# At each step, shift the decision boundary: X1 > shift => class 1
# shifts = np.linspace(0, 2, 20)  # Boundary shifts from 0 to 2
# For each shift, generate new data, compute accuracy of original model

# TODO: Step 4 - Plot accuracy over time as boundary shifts
# Show how the fixed model degrades as the concept changes

print('Exercise 2: Implement concept drift simulation above.')
print('Expected output: A plot showing accuracy degradation as the decision boundary shifts.')

### Exercise 3: Model Registry with Version Comparison
### 練習 3：模型注冊管理與版本比較

Build a `ModelRegistry` class that:
- Registers models with versions and stages (Staging/Production/Archived)
- Supports stage transitions
- Compares two model versions on test data

In [ ]:
# === Exercise 3: Model Registry ===
class ModelRegistry:
    """Simple model registry with version management."""
    def __init__(self):
        self.models = {}  # {name: {version: {'model': ..., 'stage': ..., 'metrics': ...}}}
    
    def register(self, name, version, model, metrics, stage='Staging'):
        """Register a model with given name, version, and stage."""
        # TODO: Store the model in self.models with its metadata
        # Stages: 'None' -> 'Staging' -> 'Production' -> 'Archived'
        pass
    
    def transition_stage(self, name, version, new_stage):
        """Transition a model version to a new stage."""
        # TODO: Update the stage of the specified model version
        # If transitioning to 'Production', archive any current production model
        pass
    
    def get_production_model(self, name):
        """Get the current production model for a given name."""
        # TODO: Return the model currently in 'Production' stage
        pass
    
    def compare_versions(self, name, v1, v2, X_test, y_test):
        """Compare two versions on test data, print metrics side by side."""
        # TODO: Load both versions, compute predictions, compare metrics
        pass

# Test your implementation
# registry = ModelRegistry()
# registry.register('wine-clf', '1.0', baseline_model, {'accuracy': test_acc})
# registry.register('wine-clf', '2.0', rf_tuned, {'accuracy': rf_tuned.score(X_test, y_test)})
# registry.transition_stage('wine-clf', '2.0', 'Production')
# registry.compare_versions('wine-clf', '1.0', '2.0', X_test, y_test)
print('Exercise 3: Implement the ModelRegistry class methods above.')

---
## Summary / 教學重點整理

### Key Takeaways / 關鍵要點

| 主題 Topic | 重點 Key Point |
|------|------|
| ML Lifecycle | ML 生命週期是循環的，包含問題定義→訓練→部署→監測→改善 |
| Serialization | 使用 joblib/pickle 儲存模型，採用版本命名規範 |
| Experiment Tracking | 記錄每次實驗的參數、指標、時間戳記 |
| Model Comparison | 訓練多個模型，使用交叉驗證進行公平比較 |
| Inference API | 將模型封裝為推論服務，回傳預測與信心分數 |
| Data Drift | 使用 KS 檢定偵測特徵分布變化 |
| Model Monitoring | 持續監測準確率、漂移分數、預測量 |
| A/B Testing | 使用統計檢定比較兩個模型版本 |
| CI/CD for ML | 自動化流程包含 CI、CT、CD、CM |

### MLOps 核心公式 Core Formulas

**KS Test Statistic:**
$$D = \sup_x |F_{\text{train}}(x) - F_{\text{prod}}(x)|$$

**A/B Test Z-statistic:**
$$z = \frac{\hat{p}_B - \hat{p}_A}{\sqrt{\hat{p}(1-\hat{p})\left(\frac{1}{n_A}+\frac{1}{n_B}\right)}}$$

---

### Next Week Preview / 下週預告

**Week 17: LLM & Embeddings (RAG, Prompt Engineering)**
- 文字嵌入 (Text Embeddings) 與向量空間
- TF-IDF 與語義搜尋 (Semantic Search)
- 檢索增強生成 (RAG) 概念
- 提示工程 (Prompt Engineering) 基礎